In [1]:
%pip uninstall --yes 'keras' 'matplotlib' 'scikit-learn' 'tensorflow'

Found existing installation: keras 3.10.0
Uninstalling keras-3.10.0:
  Successfully uninstalled keras-3.10.0
Found existing installation: matplotlib 3.10.0
Uninstalling matplotlib-3.10.0:
  Successfully uninstalled matplotlib-3.10.0
Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1
Found existing installation: tensorflow 2.19.0
Uninstalling tensorflow-2.19.0:
  Successfully uninstalled tensorflow-2.19.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings
warnings.simplefilter('ignore')

In [3]:
import os
import sys
import subprocess

In [4]:
def set_env(input_archive, temp_dir):

    if not os.path.exists(temp_dir):
        os.makedirs(temp_dir, exist_ok=True)
        
        subprocess.run(['tar', '-xzf', input_archive, '-C', temp_dir], check=True)
    
    subprocess.run([
        sys.executable, 
        '-m', 
        'pip', 
        'install', 
        '--no-index', 
        '--find-links', 
        f'{temp_dir}/wheels', 
        'unsloth', 
        'trl', 
        'vllm', 
        'openai_harmony'
    ], check=True)

In [5]:
set_env(
    input_archive='/kaggle/input/aimo-3-utils/wheels.tar.gz', 
    temp_dir='/kaggle/tmp/setup'
)

Looking in links: /kaggle/tmp/setup/wheels
Processing /kaggle/tmp/setup/wheels/unsloth-2025.12.9-py3-none-any.whl
Processing /kaggle/tmp/setup/wheels/trl-0.24.0-py3-none-any.whl
Processing /kaggle/tmp/setup/wheels/vllm-0.11.2-cp38-abi3-manylinux1_x86_64.whl
Processing /kaggle/tmp/setup/wheels/openai_harmony-0.0.8-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
Processing /kaggle/tmp/setup/wheels/unsloth_zoo-2025.12.7-py3-none-any.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/tyro-1.0.3-py3-none-any.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/xformers-0.0.33.post1-cp39-abi3-manylinux_2_28_x86_64.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/bitsandbytes-0.49.0-py3-none-manylinux_2_24_x86_64.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/datasets-4.3.0-py3-none-any.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/prometheus_fastapi_instrumentator-7.1.0-py3-none-any.whl (from vllm)
Processing /kaggle/tmp/setup/wheels/lm_format_enforcer-0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
kauldron 1.3.0 requires scikit-learn, which is not installed.
kauldron 1.3.0 requires tensorflow, which is not installed.
ydata-profiling 4.18.0 requires matplotlib<=3.10,>=3.5, which is not installed.
pyldavis 3.4.1 requires scikit-learn>=1.0.0, which is not installed.
stable-baselines3 2.1.0 requires matplotlib, which is not installed.
sentence-transformers 5.1.1 requires scikit-learn, which is not installed.
librosa 0.11.0 requires scikit-learn>=1.1.0, which is not installed.
cuml-cu12 25.6.0 requires scikit-learn>=1.5, which is not installed.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
bigframes 2.26.0 requires matplotlib>=3.7.1, which is not installed.
arviz 0.22.0 requires matplotlib>=3.8, which is not installed.
pynndescent 0.5.13 requires scikit-learn>=0.

In [6]:
subprocess.run(['ls', '/kaggle/tmp/setup/tiktoken_encodings'])

cl100k_base.tiktoken
o200k_base.tiktoken


CompletedProcess(args=['ls', '/kaggle/tmp/setup/tiktoken_encodings'], returncode=0)

In [7]:
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TRITON_PTXAS_PATH'] = '/usr/local/cuda/bin/ptxas'
os.environ['TIKTOKEN_ENCODINGS_BASE'] = '/kaggle/tmp/setup/tiktoken_encodings'

In [8]:
import gc
import re
import math
import time
import queue
import threading
import contextlib
from typing import Optional
from jupyter_client import KernelManager
from collections import Counter, defaultdict
from concurrent.futures import as_completed, ThreadPoolExecutor

import numpy as np
import pandas as pd
import polars as pl

from openai import OpenAI

from openai_harmony import (
    HarmonyEncodingName, 
    load_harmony_encoding, 
    SystemContent, 
    ReasoningEffort, 
    ToolNamespaceConfig, 
    Author, 
    Message, 
    Role, 
    TextContent, 
    Conversation
)

from transformers import set_seed
import kaggle_evaluation.aimo_3_inference_server

In [9]:
class CFG:

    system_prompt = (
        'You are a world-class International Mathematical Olympiad (IMO) competitor. '
        'The final answer must be a non-negative integer between 0 and 99999. '
        'You must place the final integer answer inside \\boxed{}.'
    )
    
    tool_prompt = (
        'Use this tool to execute Python code. '
        'The environment is a stateful Jupyter notebook. '
        'You must use print() to output results.'
    )

    preference_prompt = (
        'Use `math`, `numpy`,`sympy`,`itertools` and `collections` to solve the problem.'
    )

    served_model_name = 'gpt-oss'
    model_path = '/kaggle/input/gpt-oss-120b/transformers/default/1'
    
    kv_cache_dtype = 'fp8_e4m3'
    dtype = 'auto'

    high_problem_timeout = 900
    base_problem_timeout = 300

    notebook_limit = 17520
    server_timeout = 180

    session_timeout = 960
    jupyter_timeout = 10
    sandbox_timeout = 5

    stream_interval = 200
    context_tokens = 65536
    search_tokens = 1024
    buffer_tokens = 512
    batch_size = 256
    early_stop = 4
    attempts = 8
    workers = 16
    turns = 128
    seed = 42

    gpu_memory_utilization = 0.96
    temperature = 1.0
    min_p = 0.02

    # DeepConf parameters
    top_logprobs = 20           # Number of top logprobs to request for confidence calculation
    tail_tokens = 2048          # Number of tail tokens for tail confidence (C_tail)
    conf_filter_percent = 0.90  # Keep top 90% confident traces (filter bottom 10%)

In [10]:
set_seed(CFG.seed)

In [11]:
class AIMO3Template:

    def __init__(self):

        pass

    def get_system_content(self, system_prompt: str, tool_config: ToolNamespaceConfig) -> SystemContent:

        return (
            SystemContent.new()
            .with_model_identity(system_prompt)
            .with_reasoning_effort(reasoning_effort=ReasoningEffort.HIGH)
            .with_tools(tool_config)
        )

    def apply_chat_template(
        self, 
        system_prompt: str, 
        user_prompt: str, 
        tool_config: ToolNamespaceConfig
    ) -> list[Message]:

        system_content = self.get_system_content(system_prompt, tool_config)        
        system_message = Message.from_role_and_content(Role.SYSTEM, system_content)

        user_message = Message.from_role_and_content(Role.USER, user_prompt)

        return [system_message, user_message]

In [12]:
class AIMO3Sandbox:

    _port_lock = threading.Lock()
    _next_port = 50000

    @classmethod
    def _get_next_ports(cls, count: int = 5) -> list[int]:

        with cls._port_lock:
            ports = list(range(cls._next_port, cls._next_port + count))
            cls._next_port += count

            return ports

    def __init__(self, timeout: float):

        self._default_timeout = timeout
        self._owns_kernel = False
        self._client = None
        self._km = None
        
        ports = self._get_next_ports(5)

        env = os.environ.copy()
        env['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
        env['PYDEVD_WARN_EVALUATION_TIMEOUT'] = '0'
        env['JUPYTER_PLATFORM_DIRS'] = '1'
        env['PYTHONWARNINGS'] = 'ignore'
        env['MPLBACKEND'] = 'Agg'

        self._km = KernelManager()
        self._km.shell_port = ports[0]
        self._km.iopub_port = ports[1]
        self._km.stdin_port = ports[2]
        self._km.hb_port = ports[3]
        self._km.control_port = ports[4]

        self._km.start_kernel(env=env, extra_arguments=['--Application.log_level=CRITICAL'])

        self._client = self._km.blocking_client()
        self._client.start_channels()
        self._client.wait_for_ready(timeout=self._default_timeout)
        self._owns_kernel = True

        self.execute(
            'import math\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import numpy as np\n'
        )

    def _format_error(self, traceback: list[str]) -> str:

        clean_lines = []

        for frame in traceback:
            clean_frame = re.sub(r'\x1b\[[0-9;]*m', '', frame)

            if 'File "' in clean_frame and 'ipython-input' not in clean_frame:
                continue

            clean_lines.append(clean_frame)

        return ''.join(clean_lines)

    def execute(self, code: str, timeout: float | None = None) -> str:

        client = self._client
        effective_timeout = timeout or self._default_timeout
        
        msg_id = client.execute(
            code, 
            store_history=True, 
            allow_stdin=False, 
            stop_on_error=False
        )

        stdout_parts = []
        stderr_parts = []
        
        start_time = time.time()

        while True:
            elapsed = time.time() - start_time

            if elapsed > effective_timeout:
                self._km.interrupt_kernel()

                return f'[ERROR] Execution timed out after {effective_timeout} seconds'

            try:
                msg = client.get_iopub_msg(timeout=1.0)

            except queue.Empty:
                continue

            if msg.get('parent_header', {}).get('msg_id') != msg_id:
                continue

            msg_type = msg.get('msg_type')
            content = msg.get('content', {})

            if msg_type == 'stream':
                text = content.get('text', '')

                if content.get('name') == 'stdout':
                    stdout_parts.append(text)

                else:
                    stderr_parts.append(text)

            elif msg_type == 'error':
                traceback_list = content.get('traceback', [])

                stderr_parts.append(self._format_error(traceback_list))

            elif msg_type in {'execute_result', 'display_data'}:
                data = content.get('data', {})
                text = data.get('text/plain')

                if text:
                    stdout_parts.append(text if text.endswith('\n') else f'{text}\n')

            elif msg_type == 'status':
                if content.get('execution_state') == 'idle':
                    break

        stdout = ''.join(stdout_parts)
        stderr = ''.join(stderr_parts)

        if stderr:
            return f'{stdout.rstrip()}\n{stderr}' if stdout else stderr

        return stdout if stdout.strip() else '[WARN] No output. Use print() to see results.'

    def close(self):

        with contextlib.suppress(Exception):
            if self._client:
                self._client.stop_channels()

        if self._owns_kernel and self._km is not None:
            with contextlib.suppress(Exception):
                self._km.shutdown_kernel(now=True)

            with contextlib.suppress(Exception):
                self._km.cleanup_resources()

    def reset(self):

        self.execute('%reset -f')
        self.execute('import gc; gc.collect()')

        self.execute(
            'import math\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import numpy as np\n'
        )

    def __del__(self):

        self.close()

In [13]:
class AIMO3Tool:

    def __init__(self, local_jupyter_timeout: float, tool_prompt: str, sandbox=None):

        self._local_jupyter_timeout = local_jupyter_timeout
        self._tool_prompt = tool_prompt
        self._jupyter_session = sandbox
        
        self._owns_session = sandbox is None
        
        self._execution_lock = threading.Lock()
        self._init_lock = threading.Lock()

    def _ensure_session(self):

        if self._jupyter_session is None:
            with self._init_lock:
                if self._jupyter_session is None:
                    self._jupyter_session = AIMO3Sandbox(timeout=self._local_jupyter_timeout)

    def _ensure_last_print(self, code: str) -> str:

        lines = code.strip().split('\n')

        if not lines:
            return code

        last_line = lines[-1].strip()

        if 'print' in last_line or 'import' in last_line:
            return code

        if not last_line:
            return code

        if last_line.startswith('#'):
            return code

        lines[-1] = 'print(' + last_line + ')'

        return '\n'.join(lines)

    @property
    def instruction(self) -> str:

        return self._tool_prompt

    @property
    def tool_config(self) -> ToolNamespaceConfig:

        return ToolNamespaceConfig(
            name='python', 
            description=self.instruction, 
            tools=[]
        )

    def _make_response(self, output: str, channel: str | None = None) -> Message:

        content = TextContent(text=output)
        author = Author(role=Role.TOOL, name='python')
        message = Message(author=author, content=[content]).with_recipient('assistant')

        if channel:
            message = message.with_channel(channel)

        return message

    def process_sync_plus(self, message: Message) -> list[Message]:

        self._ensure_session()
        raw_script = message.content[0].text
        final_script = self._ensure_last_print(raw_script)

        with self._execution_lock:
            try:
                output = self._jupyter_session.execute(final_script)

            except TimeoutError as exc:
                output = f'[ERROR] {exc}'

        return [self._make_response(output, channel=message.channel)]

    def close(self):

        if self._jupyter_session is not None:
            if self._owns_session:
                self._jupyter_session.close()

            self._jupyter_session = None

    def __del__(self):

        self.close()

In [14]:
class AIMO3Solver:

    def __init__(self, cfg, port: int = 8000):

        self.cfg = cfg
        self.port = port
        self.base_url = f'http://0.0.0.0:{port}/v1'
        self.api_key = 'sk-local'
        self.template = AIMO3Template()
        self.encoding = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
        self.stop_token_ids = self.encoding.stop_tokens_for_assistant_actions()

        self._preload_model_weights()
        
        self.server_process = self._start_server()

        self.client = OpenAI(
            base_url=self.base_url, 
            api_key=self.api_key, 
            timeout=self.cfg.session_timeout
        )

        self._wait_for_server()
        self._initialize_kernels()

        self.notebook_start_time = time.time()
        self.problems_remaining = 50

    def _preload_model_weights(self) -> None:

        print(f'Loading model weights from {self.cfg.model_path} into OS Page Cache...')
        start_time = time.time()
        
        files_to_load = []
        total_size = 0

        for root, _, files in os.walk(self.cfg.model_path):
            for file_name in files:
                file_path = os.path.join(root, file_name)

                if os.path.isfile(file_path):
                    files_to_load.append(file_path)
                    total_size += os.path.getsize(file_path)

        def _read_file(path: str) -> None:

            with open(path, 'rb') as file_object:
                while file_object.read(1024 * 1024 * 1024):
                    pass

        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            list(executor.map(_read_file, files_to_load))

        elapsed = time.time() - start_time
        print(f'Processed {len(files_to_load)} files ({total_size / 1e9:.2f} GB) in {elapsed:.2f} seconds.\n')

    def _start_server(self) -> subprocess.Popen:

        cmd = [
            sys.executable, 
            '-m', 
            'vllm.entrypoints.openai.api_server', 
            '--seed', 
            str(self.cfg.seed), 
            '--model', 
            self.cfg.model_path, 
            '--served-model-name', 
            self.cfg.served_model_name, 
            '--tensor-parallel-size', 
            '1', 
            '--max-num-seqs', 
            str(self.cfg.batch_size), 
            '--gpu-memory-utilization', 
            str(self.cfg.gpu_memory_utilization), 
            '--host', 
            '0.0.0.0', 
            '--port', 
            str(self.port), 
            '--dtype', 
            self.cfg.dtype, 
            '--kv-cache-dtype', 
            self.cfg.kv_cache_dtype, 
            '--max-model-len', 
            str(self.cfg.context_tokens), 
            '--stream-interval', 
            str(self.cfg.stream_interval), 
            '--async-scheduling', 
            '--enable-prefix-caching'
        ]

        self.log_file = open('vllm_server.log', 'w')

        return subprocess.Popen(
            cmd, 
            stdout=self.log_file, 
            stderr=subprocess.STDOUT, 
            start_new_session=True
        )

    def _wait_for_server(self):

        print('Waiting for vLLM server...')
        start_time = time.time()

        for _ in range(self.cfg.server_timeout):
            return_code = self.server_process.poll()

            if return_code is not None:
                self.log_file.flush()

                with open('vllm_server.log', 'r') as log_file:
                    logs = log_file.read()

                raise RuntimeError(f'Server died with code {return_code}. Full logs:\n{logs}\n')

            try:
                self.client.models.list()
                elapsed = time.time() - start_time
                print(f'Server is ready (took {elapsed:.2f} seconds).\n')

                return

            except Exception:
                time.sleep(1)

        raise RuntimeError('Server failed to start (timeout).\n')

    def _initialize_kernels(self) -> None:

        print(f'Initializing {self.cfg.workers} persistent Jupyter kernels...')
        start_time = time.time()

        self.sandbox_pool = queue.Queue()

        def _create_sandbox():
            
            return AIMO3Sandbox(timeout=self.cfg.jupyter_timeout)

        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            futures = [executor.submit(_create_sandbox) for _ in range(self.cfg.workers)]

            for future in as_completed(futures):
                self.sandbox_pool.put(future.result())

        elapsed = time.time() - start_time
        print(f'Kernels initialized in {elapsed:.2f} seconds.\n')

    def _scan_for_answer(self, text: str) -> int | None:

        pattern = r'\\boxed\s*\{\s*([0-9,]+)\s*\}'
        matches = re.findall(pattern, text)

        if matches:
            try:
                clean_value = matches[-1].replace(',', '')
                value = int(clean_value)

                if 0 <= value <= 99999:
                    return value

            except ValueError:
                pass

        return None

    def _compute_token_confidences(self, logprobs_list: list) -> list:
        """
        Compute per-token confidences from logprobs.
        
        Per DeepConf paper (arxiv:2508.15260) and their GitHub implementation:
        conf_t = -mean(logprobs of ALL top-k tokens)
        
        This measures uncertainty: higher confidence when the model concentrates
        probability mass on fewer tokens.
        """
        token_confs = []
        for token_logprobs in logprobs_list:
            if token_logprobs and len(token_logprobs) > 0:
                # Use ALL top-k logprobs (not just alternatives)
                valid_lps = [lp for lp in token_logprobs if lp is not None]
                if valid_lps:
                    # Confidence = negative mean of all top-k logprobs
                    conf = -sum(valid_lps) / len(valid_lps)
                    token_confs.append(conf)
                else:
                    token_confs.append(0.0)
            else:
                token_confs.append(0.0)
        return token_confs

    def _compute_tail_confidence(self, token_confs: list) -> float:
        """
        Compute Tail Confidence (C_tail) from DeepConf paper.
        
        C_tail = (1/|T_tail|) * sum(C_t for t in T_tail)
        
        where T_tail is the last `tail_tokens` (e.g., 2048) tokens.
        This focuses on the final portion of the reasoning trace,
        which is critical for correct conclusions in mathematical reasoning.
        """
        if not token_confs:
            return 0.0
        
        tail_size = self.cfg.tail_tokens
        # Take the last tail_size tokens (or all if fewer)
        tail_confs = token_confs[-tail_size:] if len(token_confs) > tail_size else token_confs
        
        return sum(tail_confs) / len(tail_confs) if tail_confs else 0.0

    def _process_attempt(
        self, 
        problem: str, 
        system_prompt: str, 
        attempt_index: int, 
        stop_event: threading.Event, 
        deadline: float
    ) -> dict:

        if stop_event.is_set() or time.time() > deadline:
            return {
                'Attempt': attempt_index + 1, 
                'Answer': None, 
                'Confidence': 0.0,
                'Python Calls': 0, 
                'Python Errors': 0, 
                'Response Length': 0
            }

        local_tool = None
        sandbox = None
        python_calls = 0
        python_errors = 0
        total_tokens = 0
        final_answer = None
        
        # Collect logprobs across ALL turns for the entire trace
        all_logprobs = []

        attempt_seed = int(math.pow(self.cfg.seed + attempt_index, 2))

        try:
            sandbox = self.sandbox_pool.get(timeout=self.cfg.sandbox_timeout)

            local_tool = AIMO3Tool(
                local_jupyter_timeout=self.cfg.jupyter_timeout, 
                tool_prompt=self.cfg.tool_prompt, 
                sandbox=sandbox
            )

            encoding = self.encoding
            messages = self.template.apply_chat_template(
                system_prompt, 
                problem, 
                local_tool.tool_config
            )

            conversation = Conversation.from_messages(messages)

            for _ in range(self.cfg.turns):
                if stop_event.is_set() or time.time() > deadline:
                    break

                prompt_ids = encoding.render_conversation_for_completion(conversation, Role.ASSISTANT)
                max_tokens = self.cfg.context_tokens - len(prompt_ids)

                if max_tokens < self.cfg.buffer_tokens:
                    break

                # Create stream request with logprobs enabled for DeepConf
                stream = None
                try:
                    stream = self.client.completions.create(
                        model=self.cfg.served_model_name, 
                        temperature=self.cfg.temperature, 
                        max_tokens=max_tokens, 
                        prompt=prompt_ids, 
                        seed=attempt_seed, 
                        stream=True, 
                        logprobs=self.cfg.top_logprobs,  # Enable logprobs for DeepConf
                        extra_body={
                            'min_p': self.cfg.min_p, 
                            'stop_token_ids': self.stop_token_ids, 
                            'return_token_ids': True
                        },
                        timeout=max(0, deadline - time.time()),
                    )
                except Exception as e:
                    print(f"⚠️ Failed to create completion stream: {e}")
                    break
    
                if stream is None:
                    continue

                try:
                    token_buffer = []
                    text_chunks = []

                    for chunk in stream:
                        if stop_event.is_set() or time.time() > deadline:
                            break

                        new_tokens = chunk.choices[0].token_ids
                        new_text = chunk.choices[0].text

                        if new_tokens:
                            token_buffer.extend(new_tokens)
                            total_tokens += len(new_tokens)
                            text_chunks.append(new_text)

                        # Extract logprobs from chunk for DeepConf confidence
                        # Accumulate across ALL turns in this attempt
                        if hasattr(chunk.choices[0], 'logprobs') and chunk.choices[0].logprobs is not None:
                            logprobs_data = chunk.choices[0].logprobs
                            # Handle different logprobs formats from vLLM
                            if hasattr(logprobs_data, 'top_logprobs') and logprobs_data.top_logprobs:
                                for top_lp in logprobs_data.top_logprobs:
                                    if top_lp:
                                        # Extract ALL logprob values
                                        if isinstance(top_lp, dict):
                                            # vLLM returns dict {token_id: Logprob object}
                                            lp_values = [v.logprob if hasattr(v, 'logprob') else v for v in top_lp.values()]
                                        elif isinstance(top_lp, list):
                                            lp_values = [item.logprob if hasattr(item, 'logprob') else item for item in top_lp]
                                        else:
                                            lp_values = []
                                        if lp_values:
                                            all_logprobs.append(lp_values)
                            elif hasattr(logprobs_data, 'token_logprobs') and logprobs_data.token_logprobs:
                                # Fallback: just the sampled token logprobs
                                for lp in logprobs_data.token_logprobs:
                                    if lp is not None:
                                        all_logprobs.append([lp])

                        if '}' in new_text:
                            search_text = ''.join(text_chunks[-self.cfg.search_tokens:])
                            answer = self._scan_for_answer(search_text)

                            if answer is not None:
                                final_answer = answer
                                break

                finally:
                    stream.close()

                if final_answer is not None:
                    break

                if not token_buffer:
                    break

                new_messages = encoding.parse_messages_from_completion_tokens(token_buffer, Role.ASSISTANT)
                conversation.messages.extend(new_messages)
                last_message = new_messages[-1]

                if last_message.channel == 'final':
                    answer_text = last_message.content[0].text
                    final_answer = self._scan_for_answer(answer_text)
                    break

                if last_message.recipient == 'python':
                    python_calls += 1
                    print("🐍 Executing Python code...")
                    tool_responses = local_tool.process_sync_plus(last_message)

                    response_text = tool_responses[0].content[0].text

                    if response_text.startswith('[ERROR]') or 'Traceback' in response_text or 'Error:' in response_text:
                        python_errors += 1

                    conversation.messages.extend(tool_responses)
                    # Continue to next turn - logprobs will accumulate

        except Exception as exc:
            python_errors += 1

        finally:
            if local_tool is not None:
                local_tool.close()

            if sandbox is not None:
                sandbox.reset()
                self.sandbox_pool.put(sandbox)

        # Calculate DeepConf Tail Confidence from ALL collected logprobs across turns
        token_confs = self._compute_token_confidences(all_logprobs)
        confidence = self._compute_tail_confidence(token_confs)

        return {
            'Attempt': attempt_index + 1, 
            'Response Length': total_tokens, 
            'Python Calls': python_calls, 
            'Python Errors': python_errors, 
            'Answer': final_answer,
            'Confidence': confidence
        }

    def _select_answer(self, detailed_results: list) -> int:
        """
        DeepConf Weighted Majority Voting with 90% Confidence Filtering.
        
        Per DeepConf paper:
        1. Filter: Keep only top 90% confident traces (discard bottom 10%)
        2. Weight: Each answer is weighted by its tail confidence score
        3. Vote: Select answer with highest cumulative weighted votes
        
        V(a) = sum(C_t * I(answer(t) == a)) for t in filtered_traces
        """
        # Filter to only traces with valid answers
        valid_results = [r for r in detailed_results if r['Answer'] is not None]
        
        if not valid_results:
            print('\nNo valid answers found.')
            return 0
        
        # Apply 90% confidence filtering (discard bottom 10%)
        if len(valid_results) > 1:
            confidences = [r['Confidence'] for r in valid_results]
            # Calculate threshold at (1 - conf_filter_percent) * 100 percentile
            # For 90% filter, threshold is at 10th percentile
            threshold = np.percentile(confidences, (1 - self.cfg.conf_filter_percent) * 100)
            filtered_results = [r for r in valid_results if r['Confidence'] >= threshold]
            
            # Ensure we have at least one result
            if not filtered_results:
                filtered_results = valid_results
            
            num_filtered = len(valid_results) - len(filtered_results)
            if num_filtered > 0:
                print(f'🔍 Confidence filtering: Removed {num_filtered} low-confidence traces (threshold: {threshold:.4f})')
        else:
            filtered_results = valid_results
        
        # Collect weighted votes for each answer
        answer_weights = defaultdict(float)
        answer_counts = defaultdict(int)
        answer_calls = defaultdict(int)

        for result in filtered_results:
            answer = result['Answer']
            confidence = result.get('Confidence', 0.0)

            answer_weights[answer] += confidence
            answer_counts[answer] += 1
            answer_calls[answer] += result['Python Calls']

        # Sort by weighted votes (primary), then by count (secondary), then by calls (tertiary)
        sorted_answers = sorted(
            answer_weights.items(), 
            key=lambda item: (item[1], answer_counts[item[0]], answer_calls[item[0]]), 
            reverse=True
        )

        # Display voting results with DeepConf weighted scores
        vote_data = []
        for answer, weight in sorted_answers:
            vote_data.append((
                answer, 
                answer_counts[answer], 
                round(weight, 4),
                answer_calls[answer]
            ))

        vote_dataframe = pd.DataFrame(vote_data, columns=['Answer', 'Votes', 'Weighted Score', 'Calls'])
        display(vote_dataframe)

        final_answer = sorted_answers[0][0]
        final_votes = answer_counts[final_answer]
        final_weight = sorted_answers[0][1]
        final_calls = answer_calls[final_answer]

        print(f'\nFinal Result: {final_answer} | Votes: {final_votes} | Weighted Score: {final_weight:.4f} | Calls: {final_calls}\n')

        return final_answer

    def solve_problem(self, problem: str) -> int:
        
        problem_start_time = time.time()
        print(f'\nProblem: {problem}\n')

        user_input = f'{problem} {self.cfg.preference_prompt}'
        elapsed_global = time.time() - self.notebook_start_time
        time_left = self.cfg.notebook_limit - elapsed_global
        problems_left_others = max(0, self.problems_remaining - 1)
        reserved_time = problems_left_others * self.cfg.base_problem_timeout

        budget = time_left - reserved_time
        budget = min(budget, self.cfg.high_problem_timeout)
        budget = max(budget, self.cfg.base_problem_timeout)

        deadline = time.time() + budget

        print(f'Budget: {budget:.2f} seconds | Deadline: {deadline:.2f}\n')

        tasks = []

        for attempt_index in range(self.cfg.attempts):
            tasks.append((self.cfg.system_prompt, attempt_index))

        detailed_results = []
        valid_answers = []

        stop_event = threading.Event()

        executor = ThreadPoolExecutor(max_workers=self.cfg.workers)

        try:
            futures = []

            for (system_prompt, attempt_index) in tasks:
                future = executor.submit(
                    self._process_attempt, 
                    user_input, 
                    system_prompt, 
                    attempt_index, 
                    stop_event, 
                    deadline
                )

                futures.append(future)

            for future in as_completed(futures):
                try:
                    result = future.result()
                    detailed_results.append(result)

                    if result['Answer'] is not None:
                        valid_answers.append(result['Answer'])

                    counts = Counter(valid_answers).most_common(1)

                    if counts and counts[0][1] >= self.cfg.early_stop:
                        stop_event.set()

                        for f in futures:
                            f.cancel()

                        break

                except Exception as exc:
                    print(f'Future failed: {exc}')
                    continue

        finally:
            executor.shutdown(wait=False, cancel_futures=True)
            self.problems_remaining = max(0, self.problems_remaining - 1)

        # Print the inference time and budget
        used_time = time.time() - problem_start_time
        saved_time = max(0.0, budget - used_time)
        print(f"[Budget]: {budget:.2f}s\n")
        print(f"[Inference] Took {used_time:.2f}s\n")
        print(f"[Saved time]: {saved_time:.2f}s\n")

        if detailed_results:
            results_dataframe = pd.DataFrame(detailed_results)
            results_dataframe['Answer'] = results_dataframe['Answer'].astype('Int64')
            results_dataframe['Confidence'] = results_dataframe['Confidence'].round(4)
            display(results_dataframe)

        if not valid_answers:
            print('\nResult: 0\n')

            return 0

        return self._select_answer(detailed_results)

    def __del__(self):

        if hasattr(self, 'server_process'):
            self.server_process.terminate()
            self.server_process.wait()

        if hasattr(self, 'log_file'):
            self.log_file.close()

        if hasattr(self, 'sandbox_pool'):
            while not self.sandbox_pool.empty():
                try:
                    sb = self.sandbox_pool.get_nowait()
                    sb.close()

                except Exception:
                    pass

In [15]:
solver = AIMO3Solver(CFG)

Loading model weights from /kaggle/input/gpt-oss-120b/transformers/default/1 into OS Page Cache...
Processed 26 files (65.28 GB) in 91.86 seconds.

Waiting for vLLM server...
Server is ready (took 134.38 seconds).

Initializing 16 persistent Jupyter kernels...
Kernels initialized in 3.05 seconds.



In [16]:
def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    global correct_count, total_count, predictions
    
    question_id = id_.item(0)
    question_text = question.item(0)
    
    print("------")
    print(f"ID: {question_id}")
    print(f"Question: {question_text[:200]}...")
    
    final_answer = solver.solve_problem(question_text)
    predictions[question_id] = final_answer

    # Check accuracy if ground truth available
    total_count += 1
    if question_id in ground_truth:
        gt = ground_truth[question_id]
        is_correct = (final_answer == gt)
        if is_correct:
            correct_count += 1
        status = "✅" if is_correct else "❌"
        print(f"Answer: {final_answer} | Ground Truth: {gt} | {status}")
        print(f"📊 Running Accuracy: {correct_count}/{total_count} ({100*correct_count/total_count:.1f}%)")
    else:
        print(f"Answer: {final_answer}")
    
    print("------\n")
    
    return pl.DataFrame({'id': question_id, 'answer': final_answer})

In [17]:
# # Load reference data and keep ground truth for accuracy calculation
# df = pd.read_csv(
#     "/kaggle/input/ai-mathematical-olympiad-progress-prize-3/reference.csv"
# )

# # Store ground truth answers for accuracy calculation (only in local mode)
# ground_truth = dict(zip(df["id"], df["answer"])) if "answer" in df.columns else {}

# # Create input file without answers
# df.drop("answer", axis=1, errors="ignore").to_csv("reference.csv", index=False)

# # Track predictions for accuracy calculation
# predictions = {}
# correct_count = 0
# total_count = 0

In [18]:
# Load reference data and keep ground truth for accuracy calculation
df = pd.read_csv(
    "/kaggle/input/omni-math-hardestdifficulty-9/omni_math_hard.csv"
)

# df = df[df['id']==12].copy()

df = df[['id','problem','answer']]

# Store ground truth answers for accuracy calculation (only in local mode)
ground_truth = dict(zip(df["id"], df["answer"])) if "answer" in df.columns else {}

# Create input file without answers
df.drop("answer", axis=1, errors="ignore").to_csv("reference.csv", index=False)

# Track predictions for accuracy calculation
predictions = {}
correct_count = 0
total_count = 0

print(f"Dataset prepared with {len(df)} problems.")

Dataset prepared with 28 problems.


In [19]:
inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
    
else:
    inference_server.run_local_gateway(("reference.csv",))
    #inference_server.run_local_gateway(
    #    ('/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv',)
    #)

------
ID: 1797
Question: There are 60 empty boxes $B_1,\ldots,B_{60}$ in a row on a table and an unlimited supply of pebbles. Given a positive integer $n$, Alice and Bob play the following game.
In the first round, Alice take...

Problem: There are 60 empty boxes $B_1,\ldots,B_{60}$ in a row on a table and an unlimited supply of pebbles. Given a positive integer $n$, Alice and Bob play the following game.
In the first round, Alice takes $n$ pebbles and distributes them into the 60 boxes as she wishes. Each subsequent round consists of two steps:
(a) Bob chooses an integer $k$ with $1\leq k\leq 59$ and splits the boxes into the two groups $B_1,\ldots,B_k$ and $B_{k+1},\ldots,B_{60}$.
(b) Alice picks one of these two groups, adds one pebble to each box in that group, and removes one pebble from each box in the other group.
Bob wins if, at the end of any round, some box contains no pebbles. Find the smallest $n$ such that Alice can prevent Bob from winning.

[i]Czech Republic[/i]

Budget

,Attempt,Response Length,Python Calls,Python Errors,Answer,Confidence
0,1,23601,0,0,148,8.0609
1,7,32148,3,0,960,13.4672
2,2,33397,10,1,960,9.4880
3,8,36022,14,4,236,9.9357
4,5,31891,12,4,960,13.4547
5,3,35759,16,4,960,8.7370


🔍 Confidence filtering: Removed 1 low-confidence traces (threshold: 8.3990)


,Answer,Votes,Weighted Score,Calls
0,960,4,45.1470,41
1,236,1,9.9357,14



Final Result: 960 | Votes: 4 | Weighted Score: 45.1470 | Calls: 41

Answer: 960 | Ground Truth: 960 | ✅
📊 Running Accuracy: 1/1 (100.0%)
------

------
ID: 21
Question: Define the sequences $(a_n),(b_n)$ by
\begin{align*}
& a_n, b_n > 0, \forall n\in\mathbb{N_+} \\ 
& a_{n+1} = a_n - \frac{1}{1+\sum_{i=1}^n\frac{1}{a_i}} \\ 
& b_{n+1} = b_n + \frac{1}{1+\sum_{i=1}^n\...

Problem: Define the sequences $(a_n),(b_n)$ by
\begin{align*}
& a_n, b_n > 0, \forall n\in\mathbb{N_+} \\ 
& a_{n+1} = a_n - \frac{1}{1+\sum_{i=1}^n\frac{1}{a_i}} \\ 
& b_{n+1} = b_n + \frac{1}{1+\sum_{i=1}^n\frac{1}{b_i}}
\end{align*}
1) If $a_{100}b_{100} = a_{101}b_{101}$, find the value of $a_1-b_1$;
2) If $a_{100} = b_{99}$, determine which is larger between $a_{100}+b_{100}$ and $a_{101}+b_{101}$.

Budget: 900.00 seconds | Deadline: 1769429683.17

🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Exe

...


Final Result: 2500 | Votes: 3 | Weighted Score: 32.2437 | Calls: 25

Answer: 2500 | Ground Truth: 3822 | ❌
📊 Running Accuracy: 19/27 (70.4%)
------

------
ID: 1783
Question: What is the smallest positive integer $t$ such that there exist integers $x_1,x_2,\ldots,x_t$ with  \[x^3_1+x^3_2+\,\ldots\,+x^3_t=2002^{2002}\,?\]...

Problem: What is the smallest positive integer $t$ such that there exist integers $x_1,x_2,\ldots,x_t$ with  \[x^3_1+x^3_2+\,\ldots\,+x^3_t=2002^{2002}\,?\]

Budget: 900.00 seconds | Deadline: 1769437171.46

🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Exe

,Attempt,Response Length,Python Calls,Python Errors,Answer,Confidence
0,7,3816,5,0,4,9.7327
1,2,5986,6,0,4,12.2432
2,1,6428,3,0,4,11.7741
3,3,7519,4,0,4,11.4443


🔍 Confidence filtering: Removed 1 low-confidence traces (threshold: 10.2462)


,Answer,Votes,Weighted Score,Calls
0,4,3,35.4616,13



Final Result: 4 | Votes: 3 | Weighted Score: 35.4616 | Calls: 13

Answer: 4 | Ground Truth: 4 | ✅
📊 Running Accuracy: 20/28 (71.4%)
------

